<a href="https://colab.research.google.com/github/fdrecabarren/fdrecabarren/blob/main/kiva_loans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**1. Importar datos**

In [19]:
import pandas as pd
from google.colab import auth
auth.authenticate_user()

import io
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

drive_service = build('drive', 'v3')
file_id = '1U29-oHdujD5M9K17VYYIbE7RiTWUmkZN'  # kiva_loans.csv (compartido en Drive)

request = drive_service.files().get_media(fileId=file_id)
fh = io.BytesIO()
downloader = MediaIoBaseDownload(fh, request)
done = False
while not done:
    status, done = downloader.next_chunk()

fh.seek(0)
df = pd.read_csv(fh)
df.head()

,id,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,partner_id,posted_time,disbursed_time,funded_time,term_in_months,lender_count,tags,borrower_genders,repayment_interval,date
0,653051,300.0,300.0,Fruits & Vegetables,Food,"To buy seasonal, fresh fruits to sell.",PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:12:39+00:00,2013-12-17 08:00:00+00:00,2014-01-02 10:06:32+00:00,12.0,12,NaN,female,irregular,2014-01-01
1,653053,575.0,575.0,Rickshaw,Transportation,to repair and maintain the auto rickshaw used ...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:51:08+00:00,2013-12-17 08:00:00+00:00,2014-01-02 09:17:23+00:00,11.0,14,NaN,"female, female",irregular,2014-01-01
2,653068,150.0,150.0,Transportation,Transportation,To repair their old cycle-van and buy another ...,IN,India,Maynaguri,INR,334.0,2014-01-01 09:58:07+00:00,2013-12-17 08:00:00+00:00,2014-01-01 16:01:36+00:00,43.0,6,"user_favorite, user_favorite",female,bullet,2014-01-01
3,653063,200.0,200.0,Embroidery,Arts,to purchase an embroidery machine and a variet...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 08:03:11+00:00,2013-12-24 08:00:00+00:00,2014-01-01 13:00:00+00:00,11.0,8,NaN,female,irregular,2014-01-01
4,653084,400.0,400.0,Milk Sales,Food,to purchase one buffalo.,PK,Pakistan,Abdul Hakeem,PKR,245.0,2014-01-01 11:53:19+00:00,2013-12-17 08:00:00+00:00,2014-01-01 19:18:51+00:00,14.0,16,NaN,female,monthly,2014-01-01


In [20]:
df.shape

(671205, 20)

**Observación:** el dataset original contiene 671.205 registros (préstamos) y 20 columnas. Este número se usará como referencia para comparar contra el dataset final, después de la limpieza.

## **2. Análisis exploratorio rápido**

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 671205 entries, 0 to 671204
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   id                  671205 non-null  int64  
 1   funded_amount       671205 non-null  float64
 2   loan_amount         671205 non-null  float64
 3   activity            671205 non-null  object 
 4   sector              671205 non-null  object 
 5   use                 666973 non-null  object 
 6   country_code        671197 non-null  object 
 7   country             671205 non-null  object 
 8   region              614405 non-null  object 
 9   currency            671205 non-null  object 
 10  partner_id          657698 non-null  float64
 11  posted_time         671205 non-null  object 
 12  disbursed_time      668809 non-null  object 
 13  funded_time         622874 non-null  object 
 14  term_in_months      671205 non-null  float64
 15  lender_count        671205 non-nul

**Observaciones:**
- Las columnas `posted_time`, `disbursed_time`, `funded_time` y `date` están tipadas como texto (`object`) en vez de fecha. Se convertirán a `datetime` en la sección de transformaciones.
- Hay valores faltantes en varias columnas, siendo `tags` la más afectada (~171.000 valores faltantes, ~25% del dataset). Se analizará cada caso en la sección de diagnóstico.

In [22]:
df.describe()

,id,funded_amount,loan_amount,partner_id,term_in_months,lender_count
count,6.712050e+05,671205.000000,671205.000000,657698.000000,671205.000000,671205.000000
mean,9.932486e+05,785.995061,842.397107,178.199616,13.739022,20.590922
std,1.966113e+05,1130.398941,1198.660073,94.247581,8.598919,28.459551
min,6.530470e+05,0.000000,25.000000,9.000000,1.000000,0.000000
25%,8.230720e+05,250.000000,275.000000,126.000000,8.000000,7.000000
50%,9.927800e+05,450.000000,500.000000,145.000000,13.000000,13.000000
75%,1.163653e+06,900.000000,1000.000000,204.000000,14.000000,24.000000
max,1.340339e+06,100000.000000,100000.000000,536.000000,158.000000,2986.000000


**Observación:** en `loan_amount` y `funded_amount`, el percentil 75 no supera los $1.000, pero el valor máximo llega a $100.000 — una diferencia muy grande que sugiere posibles outliers. Se investigarán en detalle a continuación.

##**3. Diagnóstico de problemas**

In [23]:
df.sort_values(by="loan_amount", ascending=False).head(10)

,id,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,partner_id,posted_time,disbursed_time,funded_time,term_in_months,lender_count,tags,borrower_genders,repayment_interval,date
70499,722883,100000.0,100000.0,Agriculture,Agriculture,create more than 300 jobs for women and farmer...,HT,Haiti,Les Cayes,USD,315.0,2014-06-10 19:25:02+00:00,2014-09-08 07:00:00+00:00,2014-06-19 20:21:04+00:00,75.0,2986,"user_favorite, user_favorite, user_favorite, u...",female,irregular,2014-06-10
496715,1156972,50000.0,50000.0,Agriculture,Agriculture,"to fund the harvest of seeds of 6,000 smallhol...",MG,Madagascar,Tsihombe,USD,313.0,2016-09-24 09:08:14+00:00,2016-10-28 07:00:00+00:00,2016-11-10 09:34:50+00:00,22.0,1606,"user_favorite, user_favorite, user_favorite, u...",male,irregular,2016-09-24
408465,1055190,50000.0,50000.0,Construction,Construction,NaN,PE,Peru,NaN,USD,429.0,2016-04-15 21:35:35+00:00,2016-05-31 07:00:00+00:00,2016-05-17 16:30:32+00:00,73.0,1310,"#Eco-friendly, #Job Creator, #Biz Durable Asse...",NaN,irregular,2016-04-15
163727,812995,50000.0,50000.0,Agriculture,Agriculture,to fund its growing loan book and further deve...,KE,Kenya,Nairobi,USD,156.0,2014-12-10 16:25:04+00:00,2015-01-01 08:00:00+00:00,2015-01-19 01:22:44+00:00,38.0,1343,"user_favorite, user_favorite, user_favorite, u...",female,bullet,2014-12-10
490191,1150277,50000.0,50000.0,Health,Health,To purchase raw materials in order to produce ...,GH,Ghana,Accra,USD,489.0,2016-09-14 13:03:24+00:00,2016-11-30 08:00:00+00:00,2016-09-19 20:06:54+00:00,14.0,1569,"#Health and Sanitation, #Biz Durable Asset, #E...",male,irregular,2016-09-14
507237,1167661,9975.0,50000.0,Fruits & Vegetables,Food,to generate income opportunities for 1250+ low...,PE,Peru,Lima,USD,429.0,2016-10-13 19:32:42+00:00,2016-11-11 08:00:00+00:00,NaN,60.0,224,"user_favorite, user_favorite, user_favorite, u...",male,irregular,2016-10-13
509048,1169175,50000.0,50000.0,Poultry,Agriculture,to purchase chicken feed & a delivery vehicle ...,TZ,Tanzania,Dar es Salaam,USD,497.0,2016-10-17 13:14:54+00:00,2016-12-31 08:00:00+00:00,2016-10-21 15:29:16+00:00,14.0,1765,"#Animals, #Woman Owned Biz, #Job Creator, #Biz...",female,irregular,2016-10-17
43182,695450,50000.0,50000.0,Renewable Energy Products,Retail,To buy and sell Barefoot Power's solar lightin...,KE,Kenya,Nairobi,USD,212.0,2014-04-09 08:25:02+00:00,2014-06-09 07:00:00+00:00,2014-04-19 19:27:30+00:00,16.0,1491,"user_favorite, user_favorite, user_favorite, u...",male,bullet,2014-04-09
126839,777718,50000.0,50000.0,Agriculture,Agriculture,to buy and plant resin producing pine trees. T...,MX,Mexico,Cherán,USD,376.0,2014-10-01 20:46:15+00:00,2014-08-31 07:00:00+00:00,2014-12-07 17:02:10+00:00,144.0,586,"user_favorite, user_favorite, #Biz Durable Ass...","male, male, male, male, male, male, male, female",irregular,2014-10-01
565733,1226382,50000.0,50000.0,Agriculture,Agriculture,to pay 600 farming families 100% above market ...,EC,Ecuador,Quito,USD,514.0,2017-01-25 00:25:00+00:00,2017-03-31 07:00:00+00:00,2017-02-21 10:49:58+00:00,14.0,1689,"user_favorite, user_favorite, user_favorite, u...",female,irregular,2017-01-25


**Observación:** los préstamos más altos (entre $50.000 y $100.000) corresponden a proyectos comunitarios/agrícolas reales, con descripciones detalladas y coherentes (Haití, Tanzania, Guatemala, Sudán del Sur, etc.). No parecen errores de carga de datos, sino préstamos legítimos de mayor escala. **Decisión: se conservan.**

In [24]:
df[df["funded_amount"] < df["loan_amount"]]["funded_time"].isnull().mean()

np.float64(1.0)

**Observación:** se confirma que el 100% de los préstamos con `funded_amount` menor a `loan_amount` (es decir, no financiados completamente) tienen `funded_time` vacío. Esto no es un error de carga: no existe una fecha de financiamiento completo porque el préstamo nunca alcanzó su meta. **Decisión:** estos valores nulos se dejarán como están (o se marcarán explícitamente), ya que representan información real ("préstamo aún no financiado"), no un dato faltante por error.

In [25]:
df.duplicated().sum()

np.int64(0)

**Observación:** no se detectaron filas completamente duplicadas (`df.duplicated().sum()` = 0).

In [26]:
df["id"].duplicated().sum()

np.int64(0)

**Observación:** la columna `id` no tiene valores duplicados (`df["id"].duplicated().sum()` = 0), confirmando que cada préstamo tiene un identificador único. Combinado con el chequeo anterior (sin filas completamente duplicadas), se concluye que este dataset no presenta problemas de duplicación.

In [27]:
df.isnull().sum().sort_values(ascending=False)

,0
tags,171416
region,56800
funded_time,48331
partner_id,13507
use,4232
borrower_genders,4221
disbursed_time,2396
country_code,8
activity,0
loan_amount,0


**Observación — resumen de valores faltantes (de mayor a menor):**
- `tags`: 171.416 (~25.5% del dataset) — la más afectada
- `region`: 56.800 (~8.5%)
- `funded_time`: 48.331 (~7.2%) — ya explicado: corresponde a préstamos no financiados completamente
- `partner_id`: 13.507 (~2%)
- `use`: 4.232 (~0.6%)
- `borrower_genders`: 4.221 (~0.6%)
- `disbursed_time`: 2.396 (~0.4%)
- `country_code`: 8 (insignificante)

In [28]:
df["tags"].head(10)

,tags
0,NaN
1,NaN
2,"user_favorite, user_favorite"
3,NaN
4,NaN
5,NaN
6,"user_favorite, user_favorite"
7,"#Elderly, #Woman Owned Biz"
8,user_favorite
9,NaN


In [29]:
df["tags"] = df["tags"].fillna("sin etiquetas")
df[df["tags"] == "sin etiquetas"].head()


,id,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,partner_id,posted_time,disbursed_time,funded_time,term_in_months,lender_count,tags,borrower_genders,repayment_interval,date
0,653051,300.0,300.0,Fruits & Vegetables,Food,"To buy seasonal, fresh fruits to sell.",PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:12:39+00:00,2013-12-17 08:00:00+00:00,2014-01-02 10:06:32+00:00,12.0,12,sin etiquetas,female,irregular,2014-01-01
1,653053,575.0,575.0,Rickshaw,Transportation,to repair and maintain the auto rickshaw used ...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:51:08+00:00,2013-12-17 08:00:00+00:00,2014-01-02 09:17:23+00:00,11.0,14,sin etiquetas,"female, female",irregular,2014-01-01
3,653063,200.0,200.0,Embroidery,Arts,to purchase an embroidery machine and a variet...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 08:03:11+00:00,2013-12-24 08:00:00+00:00,2014-01-01 13:00:00+00:00,11.0,8,sin etiquetas,female,irregular,2014-01-01
4,653084,400.0,400.0,Milk Sales,Food,to purchase one buffalo.,PK,Pakistan,Abdul Hakeem,PKR,245.0,2014-01-01 11:53:19+00:00,2013-12-17 08:00:00+00:00,2014-01-01 19:18:51+00:00,14.0,16,sin etiquetas,female,monthly,2014-01-01
5,1080148,250.0,250.0,Services,Services,purchase leather for my business using ksh 20000.,KE,Kenya,NaN,KES,NaN,2014-01-01 10:06:19+00:00,2014-01-30 01:42:48+00:00,2014-01-29 14:14:57+00:00,4.0,6,sin etiquetas,female,irregular,2014-01-01


## **4. Transformaciones y limpieza**

In [30]:
df["tags"] = df["tags"].fillna("sin etiquetas")

In [31]:
df["tags"].isnull().sum()

np.int64(0)

**Observación:** la columna `tags` contenía valores nulos en los préstamos que no tenían etiquetas descriptivas asignadas (por ejemplo: #Woman Owned Biz, #Elderly, etc.). No se trata de un error de carga, sino de la ausencia real de ese dato para esa fila. Se decidió rellenar los nulos con el valor explícito `"sin etiquetas"` en lugar de eliminarlos, para no perder información de esas filas y dejar constancia clara de que no tenían etiquetas asignadas. Verificación: `df["tags"].isnull().sum()` devuelve `0`, confirmando que no quedan valores nulos en la columna.

In [32]:
df["region"] = df["region"].fillna("No especificado")
df["region"].isnull().sum()

np.int64(0)

**Observación:** la columna `region` presentaba valores nulos en los préstamos donde el país estaba registrado pero no se especificaba la región/localidad exacta. Al igual que con `tags`, no se trata de un error de carga sino de un dato descriptivo faltante. Se decidió rellenar los nulos con el valor explícito `"No especificado"` para conservar esas filas sin perder información. Verificación: `df["region"].isnull().sum()` devuelve `0`.

In [33]:
df[df["partner_id"].isnull()]["country"].value_counts().head(10)

,count
country,
Kenya,8372
United States,5064
Puerto Rico,68
Virgin Islands,2
Guam,1


**Observación:** la columna `partner_id` presentaba nulos concentrados principalmente en Kenia y Estados Unidos, lo que indica un patrón real (no aleatorio): en esos países los préstamos no se gestionan a través de un socio de campo (field partner). Por tratarse de un ID numérico, no se rellenó con texto ni con un valor arbitrario, ya que eso podría interpretarse como un ID válido. Se decidió conservar los nulos tal cual, dado que representan información legítima (ausencia de partner asignado).

In [34]:
df["use"] = df["use"].fillna("No especificado")
df["use"].isnull().sum()

np.int64(0)

**Observación:** la columna `use` (descripción textual del destino del préstamo) presentaba valores nulos en los casos donde no se registró esa descripción. Al ser texto libre, se aplicó el mismo criterio que en `tags` y `region`: se rellenaron los nulos con el valor explícito `"No especificado"` para conservar la fila sin perder información. Verificación: `df["use"].isnull().sum()` devuelve `0`.

In [35]:
df["borrower_genders"] = df["borrower_genders"].fillna("No especificado")
df["borrower_genders"].isnull().sum()

np.int64(0)

**Observación:** la columna `borrower_genders` (género de cada prestatario, con múltiples valores en préstamos grupales) presentaba nulos donde no se registró ese dato. Se aplicó el mismo criterio usado en las columnas de texto anteriores: se rellenaron los nulos con `"No especificado"` para conservar la fila. Verificación: `df["borrower_genders"].isnull().sum()` devuelve `0`.

In [36]:
df = df.dropna(subset=["country_code"])
df["country_code"].isnull().sum()

np.int64(0)

**Observación:** la columna `country_code` presentaba una cantidad mínima de valores nulos (8 filas sobre 671.205, menos del 0.01%). Al ser un código categórico usado para análisis geográficos, no se rellenó con un valor arbitrario. Dado el volumen insignificante de filas afectadas, se optó por eliminarlas directamente con `dropna(subset=["country_code"])`, evitando introducir datos inventados. Verificación: `df["country_code"].isnull().sum()` devuelve `0`.

In [37]:
df.shape

(671197, 20)

**Observación:** tras eliminar las 8 filas con `country_code` nulo, el dataset pasó de 671.205 a 671.197 filas, confirmando que la operación afectó exactamente la cantidad esperada de registros.

In [38]:
df["posted_time"] = pd.to_datetime(df["posted_time"])
df["posted_time"].dtype

datetime64[ns, UTC]

**Observación:** la columna `posted_time` estaba almacenada como texto (`object`), lo que impedía operaciones propias de fechas (ordenar cronológicamente, calcular diferencias de tiempo, filtrar por rango). Se convirtió a tipo `datetime` con `pd.to_datetime()`. Verificación: `df["posted_time"].dtype` devuelve `datetime64[ns, UTC]`.

In [39]:
df["disbursed_time"] = pd.to_datetime(df["disbursed_time"])
df["funded_time"] = pd.to_datetime(df["funded_time"])
df["date"] = pd.to_datetime(df["date"])

df[["disbursed_time", "funded_time", "date"]].dtypes

,0
disbursed_time,"datetime64[ns, UTC]"
funded_time,"datetime64[ns, UTC]"
date,datetime64[ns]


**Observación:** las columnas `posted_time`, `disbursed_time`, `funded_time` y `date` estaban almacenadas como texto (`object`), impidiendo operaciones propias de fechas. Se convirtieron todas a tipo `datetime` con `pd.to_datetime()`. Los nulos existentes en `disbursed_time` y `funded_time` se preservaron automáticamente como `NaT` (Not a Time), sin necesidad de tratamiento adicional. Verificación: `posted_time`, `disbursed_time` y `funded_time` quedaron como `datetime64[ns, UTC]`; `date` como `datetime64[ns]` (no trae zona horaria en el dato original).

In [40]:
df["sector"].unique()

array(['Food', 'Transportation', 'Arts', 'Services', 'Agriculture',
       'Manufacturing', 'Wholesale', 'Retail', 'Clothing', 'Construction',
       'Health', 'Education', 'Personal Use', 'Housing', 'Entertainment'],
      dtype=object)

**Observación:** se revisó la columna `sector` con `.unique()` para detectar inconsistencias de formato (mayúsculas/minúsculas mezcladas, espacios extra, duplicados por escritura distinta). Los 14 valores encontrados están correctamente normalizados, sin necesidad de aplicar correcciones.

In [41]:
df["activity"].nunique()
df["activity"].unique()[:30]

array(['Fruits & Vegetables', 'Rickshaw', 'Transportation', 'Embroidery',
       'Milk Sales', 'Services', 'Dairy', 'Beauty Salon', 'Manufacturing',
       'Food Production/Sales', 'Wholesale', 'General Store',
       'Clothing Sales', 'Poultry', 'Tailoring', 'Sewing', 'Bakery',
       'Restaurant', 'Food Stall', 'Farming', 'Construction Supplies',
       'Personal Products Sales', 'Home Products Sales',
       'Natural Medicines', 'Fish Selling', 'Education provider',
       'Shoe Sales', 'Machinery Rental', 'Butcher Shop', 'Pigs'],
      dtype=object)

In [42]:
df["activity"].nunique()

163

In [43]:
df["activity"].str.strip().str.lower().nunique()

163

**Observación:** se verificó la columna `activity` (163 valores únicos) comparando `.nunique()` antes y después de normalizar con `.str.strip().str.lower()`. El resultado fue idéntico (163 en ambos casos), confirmando que no existen inconsistencias de formato (mayúsculas mezcladas o espacios extra) que dupliquen categorías. No fue necesario aplicar correcciones.

In [44]:
df["country"].nunique()

86

In [45]:
df["country"].str.strip().str.lower().nunique()

86

**Observación:** se verificó la columna `country` (86 valores únicos) con el mismo método de comparación (`.nunique()` antes y después de `.str.strip().str.lower()`). El resultado fue idéntico en ambos casos, confirmando que no hay inconsistencias de formato. No fue necesario aplicar correcciones.

In [46]:
(df[["funded_amount", "loan_amount", "term_in_months", "lender_count"]] < 0).sum()

,0
funded_amount,0
loan_amount,0
term_in_months,0
lender_count,0


In [47]:
(df["term_in_months"] == 0).sum()

np.int64(0)

In [48]:
((df["lender_count"] == 0) & (df["funded_amount"] > 0)).sum()

np.int64(0)

**Observación:** se verificó que no existan préstamos con `term_in_months` igual a 0 (plazo inválido) ni casos donde `lender_count` sea 0 mientras `funded_amount` es mayor a 0 (inconsistencia lógica: dinero financiado sin prestamistas). Ambos chequeos devolvieron `0`, confirmando la integridad numérica del dataset.

In [49]:
df["porcentaje_financiado"] = (df["funded_amount"] / df["loan_amount"]) * 100
df["porcentaje_financiado"].describe()

,porcentaje_financiado
count,671197.000000
mean,96.056227
std,15.886811
min,0.000000
25%,100.000000
50%,100.000000
75%,100.000000
max,113.333333


In [50]:
(df["porcentaje_financiado"] > 100).sum()

np.int64(2)

In [51]:
df[df["porcentaje_financiado"] > 100][["id", "funded_amount", "loan_amount", "porcentaje_financiado"]]

,id,funded_amount,loan_amount,porcentaje_financiado
277188,924149,425.0,400.0,106.250000
338159,985199,3400.0,3000.0,113.333333


**Observación:** se creó la columna derivada `porcentaje_financiado` (`funded_amount / loan_amount * 100`) para medir qué proporción de cada préstamo fue financiada. La mediana es 100% (la mayoría de los préstamos se financian por completo), con un promedio de 96%. Se detectaron 2 casos (sobre 671.197 filas) con más del 100% de financiamiento (106.25% y 113.33%), correspondientes a montos coherentes y realistas. Se interpretan como sobrefinanciamiento legítimo (aportes que superaron levemente el monto solicitado) y se conservan sin modificación.

In [52]:
df["sector"] = df["sector"].astype("category")
df["country"] = df["country"].astype("category")
df["currency"] = df["currency"].astype("category")
df["repayment_interval"] = df["repayment_interval"].astype("category")

df[["sector", "country", "currency", "repayment_interval"]].dtypes

,0
sector,category
country,category
currency,category
repayment_interval,category


**Observación:** las columnas categóricas `sector`, `country`, `currency` y `repayment_interval` se convirtieron de `object` a `category`. Se optó por este tipo de codificación (en vez de one-hot encoding) porque el objetivo del dataset es el análisis exploratorio, no el entrenamiento de un modelo de machine learning. El tipo `category` reduce el uso de memoria y optimiza las operaciones de agrupamiento (`groupby`), manteniendo los valores legibles como texto.

In [53]:
df.to_csv("kiva_loans_clean_seccion4.csv", index=False)

## **5.Validación post-limpieza**

In [54]:
df.isnull().sum().sort_values(ascending=False)

,0
funded_time,48331
partner_id,13507
disbursed_time,2396
funded_amount,0
id,0
sector,0
activity,0
loan_amount,0
use,0
region,0


**Observación:** se validó el estado final de valores nulos con `df.isnull().sum()`. Todas las columnas tratadas en la sección 4 (`tags`, `region`, `use`, `borrower_genders`, `country_code`, entre otras) quedaron en 0 nulos. Las tres columnas con nulos remanentes (`funded_time`: 48.331, `partner_id`: 13.507, `disbursed_time`: 2.396) los mantienen de forma intencional, ya que representan información válida (ausencia de dato = ausencia de partner, de financiamiento completo o de desembolso registrado), documentada previamente en cada caso. La validación confirma que la limpieza se aplicó correctamente y de forma consistente con las decisiones tomadas.

In [55]:
df.dtypes

,0
id,int64
funded_amount,float64
loan_amount,float64
activity,object
sector,category
use,object
country_code,object
country,category
region,object
currency,category


**Observación:** se validaron los tipos de datos finales con `df.dtypes`. Las columnas categóricas (`sector`, `country`, `currency`, `repayment_interval`) quedaron correctamente tipadas como `category`, y las columnas de fecha (`posted_time`, `disbursed_time`, `funded_time`, `date`) como `datetime`. Esto confirma que las transformaciones de tipo aplicadas en la sección 4 se mantienen correctamente en el dataset final.

In [56]:
df.shape

(671197, 21)

## **6. Exportación final**

In [57]:
df.to_csv("kiva_loans_clean.csv", index=False)
df.to_parquet("kiva_loans_clean.parquet", index=False)

In [58]:
import os
for f in ["kiva_loans_clean.csv", "kiva_loans_clean.parquet"]:
    print(f, round(os.path.getsize(f) / (1024*1024), 2), "MB")

kiva_loans_clean.csv 194.03 MB
kiva_loans_clean.parquet 43.96 MB
